# SurfaceTarget-CRC — phase 03 — patch 3 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.3
#
# DETAILED NORMAL-TISSUE EXPRESSION AND CELL-TYPE AUDIT
#
# Objectives:
# 1. Load validated HPA and UniProt evidence
# 2. Verify HPA source integrity
# 3. Audit IHC expression and reliability categories
# 4. Construct tissue-level expression profiles
# 5. Preserve individual cell-type observations
# 6. Quantify normal-tissue staining breadth
# 7. Compare all observations with reliability-supported data
# 8. Examine leading cell-surface candidates
# 9. Review the 20 cross-omics-supported genes
# 10. Generate a descriptive tissue-expression heatmap
# 11. Save results, provenance and checkpoint
#
# IMPORTANT:
# - No final safety scores
# - No automatic target exclusions
# - Missing IHC observations are NOT zero expression
# - Normal-tissue staining is NOT clinical toxicity
# - IHC staining is NOT antibody-accessibility evidence
# - No previous analyses are overwritten
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/hpa"

RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


INTEGRATED_PATH = (
    RESULTS /
    "surfacetarget_uniprot_topology_integration.csv"
)

IHC_PATH = (
    RESULTS /
    "hpa_candidate_normal_ihc_records.csv"
)

PREVIOUS_SUMMARY_PATH = (
    RESULTS /
    "hpa_candidate_normal_ihc_summary.csv"
)

SOURCE_PATH = (
    META /
    "hpa_source_manifest.csv"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


# HPA staining levels are ordinal
# descriptive categories, not continuous
# measurements of protein abundance.

LEVEL_SCORES = {
    "not detected": 0,
    "low": 1,
    "medium": 2,
    "high": 3
}

SCORE_LABELS = {
    0: "Not detected",
    1: "Low",
    2: "Medium",
    3: "High"
}


# HPA protein-expression reliability
# categories stronger than "Uncertain".
#
# This is a secondary evidence-stratified
# view, not a guarantee that individual
# staining observations are correct.

RELIABLE_LEVELS = {
    "enhanced",
    "supported",
    "approved"
}


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.3")
print("DETAILED NORMAL-TISSUE EXPRESSION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    INTEGRATED_PATH,
    IHC_PATH,
    PREVIOUS_SUMMARY_PATH,
    SOURCE_PATH
]

for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source = pd.read_csv(
    SOURCE_PATH,
    dtype=str,
    keep_default_na=False
)


if source["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA source records."
    )


normal_source = source[
    source["dataset"].eq("normal_ihc")
]


if len(normal_source) != 1:

    raise RuntimeError(
        "Expected exactly one HPA normal-IHC source."
    )


source_record = normal_source.iloc[0]

source_file = (
    RAW /
    Path(source_record["local_path"]).name
)


if not source_file.exists():

    raise FileNotFoundError(source_file)


actual_checksum = sha256_file(source_file)


if actual_checksum.lower() != (
    source_record["sha256"].lower()
):

    raise RuntimeError(
        "HPA normal-IHC source checksum mismatch."
    )


print("\nHPA normal-IHC archive: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD INTEGRATED CANDIDATE TABLE
# ------------------------------------------------------------

candidates = pd.read_csv(
    INTEGRATED_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched",
    "uniprot_accession",
    "uniprot_topology_category"
}


if not required_columns.issubset(
    candidates.columns
):

    raise RuntimeError(
        "Unexpected UniProt integration schema."
    )


if len(candidates) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if candidates[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate Ensembl IDs."
    )


candidates["protein_enriched"] = (
    candidates["protein_enriched"]
    .astype(str)
    .str.lower()
    .eq("true")
)


if int(candidates["protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


candidate_gene_ids = set(
    candidates["ensembl_stable_id"]
)


print("\nCandidate integration loaded.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported genes:",
    int(candidates["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 4. LOAD NORMAL-TISSUE IHC RECORDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA NORMAL-TISSUE OBSERVATIONS")
print("-" * 65)


ihc = pd.read_csv(
    IHC_PATH,
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_ihc_columns = {
    "Gene",
    "Tissue",
    "Cell type",
    "Level",
    "Reliability"
}


if not required_ihc_columns.issubset(
    ihc.columns
):

    raise RuntimeError(
        "Unexpected HPA IHC record schema."
    )


ihc["Gene"] = (
    ihc["Gene"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


for column in [
    "Tissue",
    "Cell type",
    "Level",
    "Reliability"
]:

    ihc[column] = (
        ihc[column]
        .astype(str)
        .str.strip()
    )


if not set(ihc["Gene"]).issubset(
    candidate_gene_ids
):

    raise RuntimeError(
        "IHC records contain genes outside "
        "the candidate inventory."
    )


if ihc.empty:

    raise RuntimeError(
        "HPA candidate IHC table is empty."
    )


print("\nIHC records:", len(ihc))

print(
    "Candidates with IHC observations:",
    ihc["Gene"].nunique()
)

print(
    "Normal tissues represented:",
    ihc["Tissue"].nunique()
)


# ------------------------------------------------------------
# 5. EXPRESSION LEVEL AND RELIABILITY AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IHC LEVEL AND RELIABILITY AUDIT")
print("-" * 65)


ihc["level_clean"] = (
    ihc["Level"]
    .str.lower()
)


ihc["reliability_clean"] = (
    ihc["Reliability"]
    .str.lower()
)


ihc["level_score"] = (
    ihc["level_clean"]
    .map(LEVEL_SCORES)
)


ihc["interpretable_level"] = (
    ihc["level_score"].notna()
)


ihc["reliability_supported"] = (
    ihc["reliability_clean"].isin(
        RELIABLE_LEVELS
    )
)


ihc["is_high"] = (
    ihc["level_score"].eq(3)
)


ihc["is_medium_high"] = (
    ihc["level_score"].ge(2)
)


ihc["is_detected"] = (
    ihc["level_score"].ge(1)
)


ihc["is_not_detected"] = (
    ihc["level_score"].eq(0)
)


print("\nObserved expression levels:")

print(
    ihc["level_clean"]
    .value_counts()
    .to_string()
)


print("\nObserved reliability categories:")

print(
    ihc["reliability_clean"]
    .replace("", "missing")
    .value_counts()
    .to_string()
)


nonstandard = ihc[
    ~ihc["interpretable_level"]
].copy()


print(
    "\nRecords without a standard "
    "expression-level interpretation:",
    len(nonstandard)
)


if not nonstandard.empty:

    print(
        nonstandard[
            "level_clean"
        ].value_counts().to_string()
    )


print("\nIMPORTANT:")

print(
    "Nonstandard values such as ascending "
    "or descending are preserved separately."
)

print(
    "They are not treated as high, medium, "
    "low, or undetected expression."
)


# ------------------------------------------------------------
# 6. CONSTRUCT TISSUE-LEVEL PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING TISSUE-LEVEL PROFILES")
print("-" * 65)


# Each gene can have multiple IHC records
# within one tissue because different cell
# types are annotated separately.
#
# The maximum observed level is used only
# as a descriptive tissue-level indicator.
#
# A tissue receives a "High" designation
# when at least one recorded cell type
# has high staining.


tissue_profiles = (
    ihc.groupby(
        ["Gene", "Tissue"],
        sort=True
    )
    .agg(

        n_cell_type_records=(
            "Cell type",
            "size"
        ),

        n_interpretable_records=(
            "level_score",
            "count"
        ),

        max_level_score=(
            "level_score",
            "max"
        ),

        n_high_records=(
            "is_high",
            "sum"
        ),

        n_medium_high_records=(
            "is_medium_high",
            "sum"
        ),

        n_detected_records=(
            "is_detected",
            "sum"
        ),

        n_not_detected_records=(
            "is_not_detected",
            "sum"
        ),

        n_reliability_supported_records=(
            "reliability_supported",
            "sum"
        )

    )
    .reset_index()
)


tissue_profiles["max_level_label"] = (
    tissue_profiles["max_level_score"]
    .map(SCORE_LABELS)
    .fillna("No interpretable level")
)


# ------------------------------------------------------------
# 6A. Reliability-supported subset
# ------------------------------------------------------------

reliable_records = ihc[
    ihc["reliability_supported"]
].copy()


reliable_tissue = (
    reliable_records.groupby(
        ["Gene", "Tissue"]
    )
    .agg(

        max_reliable_score=(
            "level_score",
            "max"
        ),

        reliable_interpretable_records=(
            "level_score",
            "count"
        )

    )
    .reset_index()
)


tissue_profiles = tissue_profiles.merge(

    reliable_tissue,

    on=["Gene", "Tissue"],

    how="left",

    validate="one_to_one"

)


tissue_profiles[
    "reliable_max_level_label"
] = (

    tissue_profiles[
        "max_reliable_score"
    ]

    .map(SCORE_LABELS)

    .fillna(
        "No reliability-supported interpretable record"
    )

)


print(
    "\nGene-tissue profiles:",
    len(tissue_profiles)
)


print(
    "Profiles with reliability-supported "
    "interpretable observations:",
    int(
        tissue_profiles[
            "max_reliable_score"
        ].notna().sum()
    )
)


# ------------------------------------------------------------
# 7. SUMMARIZE NORMAL-TISSUE BREADTH
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING NORMAL-TISSUE STAINING BREADTH")
print("-" * 65)


summary_records = []


for gene_id, group in tissue_profiles.groupby(
    "Gene",
    sort=False
):

    interpretable = group[
        group["max_level_score"].notna()
    ]

    high = group[
        group["max_level_score"].eq(3)
    ]

    medium_high = group[
        group["max_level_score"].ge(2)
    ]

    detected = group[
        group["max_level_score"].ge(1)
    ]

    not_detected = group[
        group["max_level_score"].eq(0)
    ]

    reliable_interpretable = group[
        group["max_reliable_score"].notna()
    ]

    reliable_medium_high = group[
        group["max_reliable_score"].ge(2)
    ]

    summary_records.append({

        "ensembl_stable_id": gene_id,

        "ihc_record_available": True,

        "hpa_tissues_with_records":
            len(group),

        "hpa_tissues_interpretable":
            len(interpretable),

        "hpa_tissues_high":
            len(high),

        "hpa_tissues_medium_high":
            len(medium_high),

        "hpa_tissues_detected":
            len(detected),

        "hpa_tissues_not_detected":
            len(not_detected),

        "hpa_tissues_reliability_supported":
            len(reliable_interpretable),

        "hpa_reliable_medium_high_tissues":
            len(reliable_medium_high),

        "hpa_high_tissue_names":
            "; ".join(
                sorted(
                    high["Tissue"].tolist()
                )
            ),

        "hpa_medium_high_tissue_names":
            "; ".join(
                sorted(
                    medium_high["Tissue"].tolist()
                )
            ),

        "hpa_reliable_medium_high_tissue_names":
            "; ".join(
                sorted(
                    reliable_medium_high["Tissue"].tolist()
                )
            )

    })


gene_summary = pd.DataFrame(
    summary_records
)


if gene_summary[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene-level IHC summaries."
    )


print(
    "\nGenes with normal-tissue summaries:",
    len(gene_summary)
)


# ------------------------------------------------------------
# 8. CROSS-VALIDATE PATCH 3.1 RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-VALIDATING PREVIOUS HPA SUMMARY")
print("-" * 65)


previous = pd.read_csv(
    PREVIOUS_SUMMARY_PATH,
    dtype={
        "ensembl_stable_id": str
    }
)


previous_required = {
    "ensembl_stable_id",
    "hpa_normal_tissues_profiled",
    "hpa_normal_tissues_medium_high"
}


if not previous_required.issubset(
    previous.columns
):

    raise RuntimeError(
        "Patch 3.1 IHC summary schema differs."
    )


validation = gene_summary.merge(

    previous[
        list(previous_required)
    ],

    on="ensembl_stable_id",

    how="outer",

    indicator=True,

    validate="one_to_one"

)


if not validation["_merge"].eq("both").all():

    raise RuntimeError(
        "IHC gene coverage changed since Patch 3.1."
    )


profiled_equal = np.array_equal(

    validation[
        "hpa_tissues_with_records"
    ].to_numpy(dtype=int),

    validation[
        "hpa_normal_tissues_profiled"
    ].to_numpy(dtype=int)

)


medium_high_equal = np.array_equal(

    validation[
        "hpa_tissues_medium_high"
    ].to_numpy(dtype=int),

    validation[
        "hpa_normal_tissues_medium_high"
    ].to_numpy(dtype=int)

)


if not profiled_equal:

    raise RuntimeError(
        "Tissue coverage differs from Patch 3.1."
    )


if not medium_high_equal:

    raise RuntimeError(
        "Medium/high tissue counts differ "
        "from Patch 3.1."
    )


print("\nCandidate IHC coverage: PASSED")

print("Tissue counts: PASSED")

print("Medium/high staining counts: PASSED")


# ------------------------------------------------------------
# 9. INTEGRATE WITH RNA, CPTAC, HPA AND UNIPROT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


integrated = candidates.merge(

    gene_summary,

    on="ensembl_stable_id",

    how="left",

    validate="one_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Candidate count changed during integration."
    )


integrated["ihc_record_available"] = (
    integrated["ihc_record_available"]
    .fillna(False)
    .astype(bool)
)


count_columns = [
    "hpa_tissues_with_records",
    "hpa_tissues_interpretable",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_tissues_detected",
    "hpa_tissues_not_detected",
    "hpa_tissues_reliability_supported",
    "hpa_reliable_medium_high_tissues"
]


for column in count_columns:

    integrated[column] = (
        integrated[column]
        .fillna(0)
        .astype(int)
    )


# ------------------------------------------------------------
# 9A. Descriptive observation categories
# ------------------------------------------------------------

# These are NOT target-risk grades.
#
# They describe what the available
# normal-tissue IHC annotations report.


def describe_normal_observations(row):

    if not row["ihc_record_available"]:

        return "No HPA normal-IHC record"

    if row["hpa_tissues_interpretable"] == 0:

        return "IHC available; levels not interpretable"

    if row["hpa_tissues_high"] > 0:

        return "High staining reported in normal tissue"

    if row["hpa_tissues_medium_high"] > 0:

        return "Medium staining reported; no high observed"

    if row["hpa_tissues_detected"] > 0:

        return "Low staining only in observed tissues"

    return "Not detected in observed annotated cell types"


integrated["normal_ihc_observation_category"] = (
    integrated.apply(
        describe_normal_observations,
        axis=1
    )
)


print("\nIntegration: PASSED")

print("Candidate genes preserved:", len(integrated))

print(
    "With normal-IHC records:",
    int(integrated["ihc_record_available"].sum())
)


print("\nNormal-tissue observation categories:")

print(
    integrated[
        "normal_ihc_observation_category"
    ]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "No IHC record does not mean "
    "no healthy-tissue expression."
)

print(
    "Observed staining breadth is not "
    "a quantitative therapeutic safety score."
)


# ------------------------------------------------------------
# 10. CROSS-OMICS NORMAL-TISSUE REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS NORMAL-TISSUE REVIEW")
print("-" * 65)


cross_omics = integrated[
    integrated["protein_enriched"]
].copy()


if len(cross_omics) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


cross_omics = cross_omics.sort_values(
    [
        "hpa_tissues_medium_high",
        "gene_name"
    ],
    ascending=[
        False,
        True
    ]
)


cross_columns = [
    "gene_name",
    "uniprot_topology_category",
    "hpa_tissues_with_records",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


print("\nCross-omics-supported proteins:")

print(
    cross_omics[
        cross_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 11. LEADING CANDIDATE REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE NORMAL-TISSUE REVIEW")
print("-" * 65)


leading = integrated[
    integrated["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "protein_enriched",
    "uniprot_cell_membrane",
    "uniprot_gpi_anchor",
    "hpa_tissues_with_records",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


print("\n")

print(
    leading[
        leading_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. EXAMINE SPECIFIC NORMAL CELL TYPES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING-GENE CELL-TYPE STAINING")
print("-" * 65)


leading_ids = set(
    leading["ensembl_stable_id"]
)


leading_ihc = ihc[
    ihc["Gene"].isin(
        leading_ids
    )
    & ihc["is_medium_high"]
].copy()


leading_ihc = leading_ihc.merge(

    leading[
        [
            "ensembl_stable_id",
            "gene_name"
        ]
    ],

    left_on="Gene",

    right_on="ensembl_stable_id",

    how="left",

    validate="many_to_one"

)


leading_ihc = leading_ihc.sort_values(
    [
        "gene_name",
        "level_score",
        "Tissue",
        "Cell type"
    ],
    ascending=[
        True,
        False,
        True,
        True
    ]
)


print(
    "\nLeading-gene medium/high "
    "normal cell-type observations:",
    len(leading_ihc)
)


for gene in [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6"
]:

    subset = leading_ihc[
        leading_ihc["gene_name"].eq(
            gene
        )
    ]

    print("\n" + "-" * 50)

    print("Gene:", gene)

    print(
        "Medium/high cell-type records:",
        len(subset)
    )

    if subset.empty:

        print(
            "No medium/high observations "
            "in the available IHC records."
        )

        continue

    print(
        subset[
            [
                "Tissue",
                "Cell type",
                "Level",
                "Reliability"
            ]
        ]
        .head(8)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 13. GENERATE NORMAL-TISSUE EXPRESSION HEATMAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING NORMAL-TISSUE HEATMAP")
print("-" * 65)


all_tissues = sorted(
    tissue_profiles["Tissue"].unique()
)


leading_lookup = dict(
    zip(
        leading["gene_name"],
        leading["ensembl_stable_id"]
    )
)


heatmap = np.full(

    (
        len(LEADING_GENES),
        len(all_tissues)
    ),

    np.nan,

    dtype=float

)


tissue_lookup = {
    tissue: index
    for index, tissue in enumerate(
        all_tissues
    )
}


for gene_index, gene_name in enumerate(
    LEADING_GENES
):

    gene_id = leading_lookup.get(
        gene_name
    )

    if gene_id is None:
        continue

    rows = tissue_profiles[
        tissue_profiles["Gene"].eq(
            gene_id
        )
    ]

    for _, row in rows.iterrows():

        tissue_index = tissue_lookup[
            row["Tissue"]
        ]

        heatmap[
            gene_index,
            tissue_index
        ] = row[
            "max_level_score"
        ]


fig, ax = plt.subplots(
    figsize=(16, 7)
)


image = ax.imshow(
    heatmap,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=3
)


ax.set_xticks(
    np.arange(len(all_tissues))
)


ax.set_xticklabels(
    all_tissues,
    rotation=90,
    fontsize=8
)


ax.set_yticks(
    np.arange(len(LEADING_GENES))
)


ax.set_yticklabels(
    LEADING_GENES
)


ax.set_title(
    "HPA Normal-Tissue IHC — "
    "Leading SurfaceTarget-CRC Candidates"
)


ax.set_xlabel(
    "Normal tissue"
)


ax.set_ylabel(
    "Candidate gene"
)


colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.025,
    pad=0.02
)


colorbar.set_ticks(
    [0, 1, 2, 3]
)


colorbar.set_ticklabels(
    [
        "Not detected",
        "Low",
        "Medium",
        "High"
    ]
)


colorbar.set_label(
    "Maximum observed cell-type staining level"
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "surfacetarget_normal_tissue_ihc_heatmap.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nNormal-tissue heatmap saved.")

print(
    "Missing heatmap values indicate "
    "unavailable or uninterpretable IHC data."
)


# ------------------------------------------------------------
# 14. SAVE ANALYSIS TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING NORMAL-TISSUE AUDIT RESULTS")
print("-" * 65)


ihc.to_csv(

    RESULTS /
    "hpa_candidate_celltype_staining_audit.csv",

    index=False

)


tissue_profiles.to_csv(

    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv",

    index=False

)


gene_summary.to_csv(

    RESULTS /
    "hpa_candidate_detailed_tissue_summary.csv",

    index=False

)


integrated.to_csv(

    RESULTS /
    "surfacetarget_normal_tissue_integration.csv",

    index=False

)


cross_omics.to_csv(

    RESULTS /
    "surfacetarget_crossomics_normal_tissue_review.csv",

    index=False

)


leading_ihc.to_csv(

    RESULTS /
    "surfacetarget_leading_normal_celltypes.csv",

    index=False

)


leading.to_csv(

    RESULTS /
    "surfacetarget_leading_normal_tissue_review.csv",

    index=False

)


nonstandard.to_csv(

    RESULTS /
    "hpa_nonstandard_ihc_level_review.csv",

    index=False

)


print("\nAll normal-tissue audit tables saved.")


# ------------------------------------------------------------
# 15. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "Human Protein Atlas",

    "source_dataset": "normal_ihc_data.tsv.zip",

    "source_sha256": actual_checksum,

    "candidate_count": int(
        len(integrated)
    ),

    "cross_omics_supported_count": int(
        len(cross_omics)
    ),

    "ihc_records": int(
        len(ihc)
    ),

    "ihc_annotated_genes": int(
        len(gene_summary)
    ),

    "gene_tissue_profiles": int(
        len(tissue_profiles)
    ),

    "normal_tissues_represented": int(
        len(all_tissues)
    ),

    "nonstandard_expression_records": int(
        len(nonstandard)
    ),

    "expression_level_mapping":
        LEVEL_SCORES,

    "reliability_supported_categories":
        sorted(RELIABLE_LEVELS),

    "tissue_level_aggregation": (
        "Maximum observed categorical IHC "
        "level across annotated cell types"
    ),

    "reliability_analysis": (
        "All observations retained, with "
        "a separate Enhanced/Supported/Approved "
        "evidence-stratified summary"
    ),

    "previous_hpa_summary_validation": True,

    "normal_tissue_safety_score_created": False,

    "therapeutic_target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "IHC expression levels are ordinal categories.",

        "Maximum tissue staining represents "
        "the highest observed annotated cell type.",

        "Tissue-level summaries can obscure "
        "important cell-type differences.",

        "HPA reliability is not a direct "
        "measure of therapeutic accessibility.",

        "Uncertain reliability records are "
        "retained rather than discarded.",

        "Uninterpretable expression labels "
        "are not converted to numeric levels.",

        "Missing IHC measurements are "
        "not treated as absent expression.",

        "Normal-tissue staining does not "
        "establish clinical toxicity.",

        "Tumor and normal measurements "
        "originate from different resources.",

        "No cross-platform quantitative "
        "tumor-normal ratios were calculated.",

        "Antibody binding and internalization "
        "have not been established.",

        "No therapeutic targets have been validated."

    ]
}


with open(

    META /
    "hpa_detailed_normal_tissue_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 16. CREATE PHASE 3 RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 3 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase3_Patch33_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.3 SUMMARY")
print("=" * 65)


print(
    "\nTranscriptomic candidates:",
    len(integrated)
)

print(
    "Cross-omics-supported genes:",
    len(cross_omics)
)

print(
    "Genes with normal-tissue IHC:",
    int(
        integrated[
            "ihc_record_available"
        ].sum()
    )
)

print(
    "Gene-tissue profiles constructed:",
    len(tissue_profiles)
)

print(
    "Individual cell-type records audited:",
    len(ihc)
)

print(
    "Nonstandard expression records preserved:",
    len(nonstandard)
)


print("\nPrevious HPA summary validation: PASSED")

print("Normal-tissue audit tables: SAVED")

print("Normal-tissue heatmap: SAVED")

print("Research provenance: SAVED")

print("Phase 3 checkpoint: CREATED")


print("\nNo final target ranking.")

print("No safety scores assigned.")

print("No candidate genes excluded.")

print("No previous analyses overwritten.")


print("\nPATCH 3.3 COMPLETE")

print("=" * 65)